# 02 · Microblogging mit MongoDB

[Demo-Übersicht](README.md) · [Technische Vorbereitung](../../README_technical_preparation.md) · [UAP-Aufgabe](../../tasks/02_mongodb/README.md)

Ein Post enthält hier seine Kommentare und den vorberechneten Like-Zähler. Wie verändert das unsere Zugriffe gegenüber SQLite?
**Der Dokumentblick und Q1 bilden den Kern; Q2–Q5 bleiben als Vergleich und Vertiefung verfügbar.**

Die Demo braucht die Kursumgebung und einen laufenden MongoDB-Server: [Docker/Codespaces](../../README_technical_preparation.md) oder [Community Server ohne Docker](../../docs/setup/MONGODB_LOKAL.md). Verwende den Kernel **Python (rothstein-storage-workshop-2026)**. Es sind keine zusätzlichen Pakete oder Cloud-Konten erforderlich.

## 1. Dokumente vorbereiten

Nutzer, Posts mit eingebetteten Kommentaren und Follows kommen aus JSONL; die vollständigen Likes aus CSV sind für Q1 erforderlich. Stabile `_id`-Werte stammen aus fachlichen IDs beziehungsweise dem gerichteten Follow-Paar.

Die synthetischen Originalzeitstempel enthalten keine Zeitzone. Für diese Demo vereinbaren wir ausdrücklich UTC, um BSON-Datumswerte und Zeitfenster eindeutig zu behandeln. Das ist eine Lehrkonvention, keine nachträglich ermittelte Quellzeitzone. Die Kalendertage bleiben mit der SQLite-Demo vergleichbar.

In [ ]:
from pathlib import Path
import sys
from copy import deepcopy
from datetime import datetime, timedelta, timezone
import json
import pandas as pd
from IPython.display import display
from pprint import pprint

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "scripts/mongodb_workshop.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Öffne das Notebook innerhalb des Repositories.")
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
from mongodb_workshop import (prepare_documents, collection_names, mongo_workspace,
                              import_snapshot, validate_note, save_note, probe_note_schema)
from storage_runtime import connection_summary
pd.set_option("display.max_colwidth", 90)
print("MongoDB-Ziel:", connection_summary()["MongoDB"])

DATASET = "microblogging"
names = collection_names(DATASET)
inputs = prepare_documents(DATASET)
display(pd.DataFrame({"Collection": inputs.keys(), "Dokumente": [len(v) for v in inputs.values()]}))
pprint(inputs["posts"][0])

## 2. Wiederholbarer Import

`ReplaceOne(..., upsert=True)` schreibt jeden Datensatz unter derselben ID. Nach erfolgreichen Upserts entfernt der Helfer überholte IDs ausschliesslich aus der jeweiligen Demo-Collection. So stellt ein erneuter Lauf den Snapshot wieder her; eigene Änderungen in diesen Collections werden überschrieben.

Der gesamte Import ist auf unserem **Standalone-Server nicht atomar**. Nach einem Abbruch kann ein Teilbestand vorliegen; wiederhole dann den Import und werte erst nach `IMPORT OK` aus. Ein einzelner Dokument-Schreibzugriff ist atomar. [MongoDB: Atomarität](https://www.mongodb.com/docs/manual/core/write-operations-atomicity/).

In [ ]:
with mongo_workspace(DATASET) as (db, names):
    first = import_snapshot(db, names, DATASET, inputs)
with mongo_workspace(DATASET) as (db, names):
    persisted = {key:db[names[key]].count_documents({}) for key in inputs}
    second = import_snapshot(db, names, DATASET, inputs)
assert first == persisted == second == {"users":200,"posts":865,"follows":3047,"likes":1266}
print("IMPORT OK: erneut verbunden und ohne Verdoppelung wiederholt geladen.")

## Dokumentblick · Post und Kommentare gemeinsam lesen

Eine Abfrage liefert Post 1 einschliesslich seiner Kommentare. `author_id` referenziert ein Nutzerkonto; dessen vollständiges Profil ist nicht in jedem Post dupliziert.

Die Einbettung passt zu diesem kleinen, begrenzten Lehrbestand. Bei unbegrenzt wachsenden Kommentaren würde das Dokument wachsen und häufiger geändert; eine produktive Plattform kann Kommentare separat speichern oder nur eine begrenzte Auswahl einbetten. [MongoDB: Einbettung](https://www.mongodb.com/docs/manual/data-modeling/concepts/embedding-vs-references/).

In [ ]:
with mongo_workspace(DATASET) as (db, names):
    post = db[names["posts"]].find_one({"_id":1})
pprint(post)
assert post["comments"][0]["comment_id"] == 1

## Q1 · Wer ist aktiv? — Kernabfrage

Engagement = **verfasste Posts + verfasste Kommentare + gegebene Likes**, gesamter Eingabebestand. Mehrere Kommentare derselben Person bleiben mehrere Interaktionen.

`$unwind` macht aus dem Kommentar-Array einzelne Pipeline-Zeilen. `$unionWith` führt die drei Aktivitätsarten zusammen; `$group` summiert pro Nutzer. Nullzeilen für alle Nutzer erhalten auch inaktive Konten. Der anschliessende `$lookup` ergänzt den Namen.

In [ ]:
# Nullzeilen für alle Nutzer, damit auch inaktive Konten erhalten bleiben.
pipeline_q1 = [
    {"$project": {"_id":"$user_id", "posts_count":{"$literal":0},
                  "comments_made":{"$literal":0}, "likes_made":{"$literal":0}}},
    {"$unionWith": {"coll":names["posts"], "pipeline":[
        {"$group":{"_id":"$author_id", "posts_count":{"$sum":1}}}]}},
    {"$unionWith": {"coll":names["posts"], "pipeline":[
        {"$unwind":"$comments"},
        {"$group":{"_id":"$comments.user_id", "comments_made":{"$sum":1}}}]}},
    {"$unionWith": {"coll":names["likes"], "pipeline":[
        {"$group":{"_id":"$user_id", "likes_made":{"$sum":1}}}]}},
    {"$group":{"_id":"$_id", "posts_count":{"$sum":"$posts_count"},
                "comments_made":{"$sum":"$comments_made"}, "likes_made":{"$sum":"$likes_made"}}},
    {"$set":{"engagement_score":{"$add":["$posts_count","$comments_made","$likes_made"]}}},
    {"$lookup":{"from":names["users"], "localField":"_id", "foreignField":"user_id", "as":"user"}},
    {"$project":{"_id":0, "user_id":"$_id", "username":{"$arrayElemAt":["$user.username",0]},
                  "posts_count":1,"comments_made":1,"likes_made":1,"engagement_score":1}},
    {"$sort":{"engagement_score":-1,"posts_count":-1,"user_id":1}}
]

In [ ]:
with mongo_workspace(DATASET) as (db, names):
    engagement = pd.DataFrame(list(db[names["users"]].aggregate(pipeline_q1)))
display(engagement.head(10))
assert len(engagement) == 200 and engagement["engagement_score"].sum() == 2643
assert engagement.head(3)["user_id"].tolist() == [94,107,35]

In [ ]:
import matplotlib.pyplot as plt
ax = engagement.head(10).sort_values("engagement_score").plot.barh(
    x="username", y=["posts_count","comments_made","likes_made"], stacked=True,
    color=["#176B87","#DFA33B","#6A7195"], figsize=(9,4.5))
ax.set(title="Aktivität im Dokumentmodell", xlabel="Ausgeführte Aktivitäten", ylabel="Nutzerkonto")
ax.legend(["Posts","Kommentare","Gegebene Likes"],loc="lower right")
plt.tight_layout()
plt.show()

## Q2 · Welche Posts erhalten Likes? — Vertiefung

`like_count` ist bereits im Post gespeichert. Das vereinfacht das Lesen, ist aber redundante Information: Bei Änderungen an Likes muss die Anwendung den Zähler mitführen oder aus den Ereignissen neu berechnen. Im festen Snapshot prüft die Vorbereitung jeden Zähler gegen die Likes-CSV.

Posts ohne Likes bleiben mit `like_count=0` erhalten. Gleichstände werden nach Post-ID sortiert.

In [ ]:
pipeline_q2 = [
    {"$sort":{"like_count":-1,"_id":1}},
    {"$lookup":{"from":names["users"],"localField":"author_id","foreignField":"user_id","as":"author"}},
    {"$project":{"_id":0,"post_id":"$_id","author":{"$arrayElemAt":["$author.username",0]},
                  "created_at":1,"like_count":1,"preview":{"$substrCP":["$text",0,80]}}}
]

In [ ]:
with mongo_workspace(DATASET) as (db, names):
    popular_posts = pd.DataFrame(list(db[names["posts"]].aggregate(pipeline_q2)))
display(popular_posts.head(10))
assert len(popular_posts)==865 and popular_posts["like_count"].sum()==1266

## Q3 · Beziehungen und begrenzte Distanz — Vertiefung

Die Richtung eines Follows bleibt `src_user_id → dst_user_id`. Zuerst zählen wir beide Richtungen pro Person.

Danach untersuchen wir wie bei SQLite **1 → 3**, begrenzt auf drei Schritte. Bei `$graphLookup` hat die erste erreichte Follow-Kante `depth=0`, bedeutet aber bereits **einen Beziehungsschritt**. Deshalb ist `maxDepth=max_hops-1` und die ausgegebene Distanz `depth+1`.

Diese Pipeline liefert eine minimale Distanz innerhalb des Suchbereichs, **keine geordnete Knotenfolge**. Ein leeres Ergebnis bedeutet nur, dass das Ziel innerhalb der Begrenzung nicht erreicht wurde. [MongoDB: $graphLookup](https://www.mongodb.com/docs/manual/reference/operator/aggregation/graphLookup/).

In [ ]:
pipeline_degrees = [
    {"$lookup":{"from":names["follows"],"localField":"user_id","foreignField":"dst_user_id","as":"incoming"}},
    {"$lookup":{"from":names["follows"],"localField":"user_id","foreignField":"src_user_id","as":"outgoing"}},
    {"$project":{"_id":0,"user_id":1,"username":1,
                  "followers":{"$size":"$incoming"},"following":{"$size":"$outgoing"}}},
    {"$sort":{"followers":-1,"following":-1,"user_id":1}}
]
src, dst, max_hops = 1, 3, 3
pipeline_distance = [
    {"$match":{"user_id":src}},
    {"$graphLookup":{"from":names["follows"], "startWith":"$user_id",
                     "connectFromField":"dst_user_id", "connectToField":"src_user_id",
                     "as":"reachable", "maxDepth":max_hops-1, "depthField":"depth"}},
    {"$unwind":"$reachable"},
    {"$match":{"reachable.dst_user_id":dst}},
    {"$group":{"_id":None,"hops":{"$min":{"$add":["$reachable.depth",1]}}}},
    {"$project":{"_id":0,"hops":1}}
]

In [ ]:
with mongo_workspace(DATASET) as (db,names):
    degrees = pd.DataFrame(list(db[names["users"]].aggregate(pipeline_degrees)))
    distance = list(db[names["users"]].aggregate(pipeline_distance))
display(degrees.head(10))
print("Distanz:",distance)
assert degrees["followers"].sum()==degrees["following"].sum()==3047
assert distance == [{"hops":2}]

## Q4 · Posts pro Kalendertag — Vertiefung

MongoDB zählt die Posts pro UTC-Tag. **pandas ergänzt danach den Kalender und berechnet den gleitenden Durchschnitt** über bis zu sieben Tage. Damit sind auch Tage ohne Posts enthalten; sieben Zeilen bedeuten sieben Kalendertage. Diese Arbeitsteilung ist bewusst gewählt und keine fehlende Fähigkeit von MongoDB.

In [ ]:
pipeline_daily = [
    {"$group":{"_id":{"$dateTrunc":{"date":"$created_at","unit":"day","timezone":"UTC"}},
                "posts":{"$sum":1}}},
    {"$project":{"_id":0,"day":"$_id","posts":1}},
    {"$sort":{"day":1}}
]

def complete_calendar(records):
    # Python/pandas-Schritt nach der datenbankseitigen Tageszählung.
    daily = pd.DataFrame(records).set_index("day").sort_index()
    daily.index = pd.to_datetime(daily.index, utc=True)
    days = pd.date_range(daily.index.min(), daily.index.max(), freq="D")
    result = daily.reindex(days, fill_value=0).rename_axis("day").reset_index()
    result["rolling_7d"] = result["posts"].rolling(7, min_periods=1).mean()
    return result

In [ ]:
with mongo_workspace(DATASET) as (db,names):
    daily_records = list(db[names["posts"]].aggregate(pipeline_daily))
trend = complete_calendar(daily_records)
display(trend.head(8))
assert len(trend)==30 and trend["posts"].sum()==865
fig, ax = plt.subplots(figsize=(9,4))
ax.bar(trend["day"],trend["posts"],color="#B6D7DF",label="Posts pro Tag")
ax.plot(trend["day"],trend["rolling_7d"],color="#176B87",linewidth=2,label="Gleitender Durchschnitt (bis zu 7 Tage)")
ax.set(title="Microblogging: Kalender und gleitender Durchschnitt in pandas",xlabel="Datum (UTC)",ylabel="Anzahl Posts")
ax.legend();fig.autofmt_xdate();plt.tight_layout();plt.show()

## Q5 · Feed aus dem festen Datenfenster — Vertiefung

Nutzer 1; Intervall **[8. September 2025, 11. September 2025)**. Berücksichtigt werden Follow-Beziehungen, die zu Beginn des Fensters schon bestanden. Eine Unfollow-Historie ist nicht vorhanden.

Der Bezugspunkt folgt aus dem letzten Post-Tag, nicht aus dem heutigen Datum. Die Datenbank liefert zuerst die Followings, dann die gefilterten Posts mit den referenzierten Nutzernamen.

In [ ]:
last_day = max(d["created_at"] for d in inputs["posts"]).replace(hour=0, minute=0, second=0, microsecond=0)
until = last_day + timedelta(days=1)
since = until - timedelta(days=3)
viewer_user_id = 1

def feed_pipeline(author_ids):
    return [
        {"$match":{"author_id":{"$in":author_ids},"created_at":{"$gte":since,"$lt":until}}},
        {"$sort":{"created_at":-1,"_id":1}},
        {"$limit":50},
        {"$lookup":{"from":names["users"],"localField":"author_id","foreignField":"user_id","as":"author"}},
        {"$project":{"_id":0,"post_id":"$_id","author":{"$arrayElemAt":["$author.username",0]},
                      "created_at":1,"like_count":1,"preview":{"$substrCP":["$text",0,100]}}}
    ]

In [ ]:
with mongo_workspace(DATASET) as (db,names):
    followings = [d["dst_user_id"] for d in db[names["follows"]].find(
        {"src_user_id":viewer_user_id,"since":{"$lte":since}}, {"_id":0,"dst_user_id":1})]
    pipeline_feed = feed_pipeline(followings)
    feed = pd.DataFrame(list(db[names["posts"]].aggregate(pipeline_feed)))
display(feed)
assert feed["post_id"].tolist()==[23,399,204,396]

## Indexblick und Transfer

Untersuche den Post-Filter aus Q5. Welche Rolle kann der Index `(author_id, created_at)` spielen? Ein zusätzliches Sortieren kann erforderlich bleiben; ein Index garantiert nicht automatisch die beste Laufzeit. `explain()` beschreibt den gewählten Plan und ist hier kein Leistungsbenchmark.

In [ ]:
with mongo_workspace(DATASET) as (db,names):
    plan = db[names["posts"]].find(pipeline_feed[0]["$match"]).sort([("created_at",-1),("_id",1)]).explain()
pprint(plan["queryPlanner"]["winningPlan"])

Welche Daten liest und ändert eine Anwendung gemeinsam? Was wird durch Einbettung einfacher, welche Redundanz entsteht?
Weiter mit der [UAP-Aufgabe](../../tasks/02_mongodb/task.ipynb): Aktenansicht, optionale Quellfelder und eigene, belegte Lesernotizen.